We have learned the basics from the `SocialMediaCommentScraper_ExploratoryNotes.ipynb`.

This notebook will run the actual production of text data.

In [1]:
import requests
import json
import pandas as pd
import matplotlib.pyplot as plt

import os
from google.colab import drive

import time
from datetime import datetime, timedelta

## Extract Reddit comments using Arctic Shift API:
https://github.com/ArthurHeitmann/arctic_shift

### Functions for processing

In [2]:
# def get_reddit_comments(symbol,  start, end, num_comments="auto"):
#     url = "https://arctic-shift.photon-reddit.com/api/comments/search"

#     params = {
#         "subreddit": "wallstreetbets",
#         "body": symbol,
#         "after": start,
#         "before": end,
#         "limit": num_comments #"auto" gives upto 1000 comments depending the server status
#     }

#     r = requests.get(url, params=params)

#     if r.status_code == 200:
#         data = r.json()
#     else:
#         data = None

#     #print("Request status (accepted: 200; rejected: 400):", r.status_code)


#     return data, r.status_code


def get_reddit_comments(symbol, start, end, num_comments="auto"):
    url = "https://arctic-shift.photon-reddit.com/api/comments/search"

    params = {
        "subreddit": "wallstreetbets",
        "body": symbol,
        "after": start,
        "before": end,
        "limit": num_comments
    }

    r = requests.get(url, params=params)

    try:
        data = r.json()
    except ValueError:
        data = None

    return data, r.status_code, r.headers

In [3]:
def clean_comments(data):
    #data is a list, so use data["data"] in this notebook.
    KEEP_COLUMNS = [
        "author",
        "body",
        "created_utc",
        "id",
        "is_submitter",
        "link_id",
        "parent_id",
        "score",
    ]

    comments_clean = [
        {
            key: comment.get(key)
            for key in KEEP_COLUMNS
        }
        for comment in data
    ]

    return comments_clean

### Environment Setup

In [4]:
# Locate google-drive folder
drive.mount(f"/content/drive/")
print(os.getcwd())

project_path = "/content/drive/MyDrive/Colab Notebooks/dl-quant_project/DataSource"

Mounted at /content/drive/
/content


### Test production of text data

In [ ]:
data, status = get_reddit_comments("GME", "2021-01-01", "2021-01-02", 100) #limit is 100?
print("Request status (accepted: 200; rejected: 400):", status)
if status == 200:
  print("Length:",len(data["data"])) #Check if we get something here

Request status (accepted: 200; rejected: 400): 200
Length: 100


In [ ]:
#This seems not working:
data = get_reddit_comments("GME", "2021-01-01", "2021-01-03") #limit is 100?

Request status (accepted: 200; rejected: 400): 400


TypeError: object of type 'NoneType' has no len()

#### Collect text data by days

In [ ]:
#manually loop over a couple days to collect more comments.
days_comments = []

for day in range(1, 6):
    start = f"2021-01-{day:02d}"
    end = f"2021-01-{day+1:02d}"

    while True:
        print(f"{datetime.now().strftime("%Y-%m-%d %H:%M:%S")} Requesting {start} → {end}...", end=" ")

        data, status = get_reddit_comments("GME", start, end, 100)

        if status == 200:
            comments = data["data"]
            days_comments.extend(comments)

            print(f"OK — {len(comments)} comments | Total: {len(days_comments)}")
            break

        else:
            print(f"failed (status {status}). Retrying in 5 seconds...")
            time.sleep(5)

print(f"\nFinished. Total comments: {len(days_comments)}")

2026-10-08 14:46:53 Requesting 2021-01-01 → 2021-01-02... failed (status 422). Retrying in 5 seconds...
2026-10-08 14:47:00 Requesting 2021-01-01 → 2021-01-02... OK — 100 comments | Total: 100
2026-10-08 14:47:05 Requesting 2021-01-02 → 2021-01-03... OK — 100 comments | Total: 200
2026-10-08 14:47:11 Requesting 2021-01-03 → 2021-01-04... OK — 100 comments | Total: 300
2026-10-08 14:47:16 Requesting 2021-01-04 → 2021-01-05... OK — 100 comments | Total: 400
2026-10-08 14:47:21 Requesting 2021-01-05 → 2021-01-06... failed (status 422). Retrying in 5 seconds...
2026-10-08 14:47:26 Requesting 2021-01-05 → 2021-01-06... failed (status 422). Retrying in 5 seconds...
2026-10-08 14:47:32 Requesting 2021-01-05 → 2021-01-06... OK — 100 comments | Total: 500

Finished. Total comments: 500


In [ ]:
comments_clean = clean_comments(days_comments)
df = pd.DataFrame(comments_clean)

# df.to_parquet(
#     f"{project_path}/wallstreetbets_GME_Jan2021.parquet",
#     index=False
# )

df.to_csv(
    f"{project_path}/wallstreetbets_GME_Jan2021_Jan1_Jan5.csv",
    index=False
)


#### Collect text data by months

In [ ]:
#Now we may want to collect more than 1 month text data:
months_comments = []

start_date = datetime(2021, 2, 1) #yyyy-mm-dd
end_date = datetime(2021, 3, 10)

current_date = start_date

while current_date < end_date:
    next_date = current_date + timedelta(days=1)

    start = current_date.strftime("%Y-%m-%d")
    end = next_date.strftime("%Y-%m-%d")

    while True:
        print(f"{datetime.now().strftime("%Y-%m-%d %H:%M:%S")} Requesting {start} → {end}...", end=" ")

        data, status = get_reddit_comments("GME", start, end, 100)

        if status == 200:
            comments = data["data"]
            months_comments.extend(comments)

            print(
                f"OK — {len(comments)} comments | "
                f"Total: {len(months_comments)}"
            )
            break

        else:
            print(f"failed (status {status}). Retrying in 10 seconds...")
            time.sleep(10)

    current_date = next_date

print(f"\nFinished. Total comments: {len(months_comments)}")

Requesting 2021-02-01 → 2021-02-02... OK — 100 comments | Total: 100
Requesting 2021-02-02 → 2021-02-03... OK — 100 comments | Total: 200
Requesting 2021-02-03 → 2021-02-04... failed (status 422). Retrying in 10 seconds...
Requesting 2021-02-03 → 2021-02-04... failed (status 422). Retrying in 10 seconds...
Requesting 2021-02-03 → 2021-02-04... failed (status 422). Retrying in 10 seconds...
Requesting 2021-02-03 → 2021-02-04... OK — 100 comments | Total: 300
Requesting 2021-02-04 → 2021-02-05... failed (status 422). Retrying in 10 seconds...
Requesting 2021-02-04 → 2021-02-05... OK — 100 comments | Total: 400
Requesting 2021-02-05 → 2021-02-06... OK — 100 comments | Total: 500
Requesting 2021-02-06 → 2021-02-07... OK — 100 comments | Total: 600
Requesting 2021-02-07 → 2021-02-08... failed (status 422). Retrying in 10 seconds...
Requesting 2021-02-07 → 2021-02-08... OK — 100 comments | Total: 700
Requesting 2021-02-08 → 2021-02-09... failed (status 422). Retrying in 10 seconds...
Request

In [ ]:
comments_clean = clean_comments(months_comments)
df = pd.DataFrame(comments_clean)

# df.to_parquet(
#     f"{project_path}/wallstreetbets_GME_Jan2021.parquet",
#     index=False
# )

df.to_csv(
    f"{project_path}/wallstreetbets_GME_Jan2021_Feb1_Mar10.csv",
    index=False
)

### Production Task

GME/AMC/BB in 2021 and SMR/SMTC in 2025

If we look at the most trading volume dates, we could narrow down to:
- GME 1/4/2021 - 4/12/2021
- AMC 1/4/2021 - 4/5/2021; 5/3/2021 - 7/5/2021
- BB 11/30/2020 - 12/9/2020; 1/13/2021 - 2/9/2021; 2/25/2021 - 6/21/2021

- SMR 2/19/2024 - 4/15/2-25; 9/2/2024 - 4/14/2025
- SMTC 3/10/2025 - 3/31/2025 (? this one looks less meme-like, as trading volume seems stable overtime)

#### Functions we need

In [5]:
# The basic collecting function, assuming error codes 400+ will clear up after a few trials
def collect_reddit_comments(symbol, start_date, end_date, num_comments=100):
    months_comments = []

    current_date = start_date

    while current_date < end_date:
        next_date = current_date + timedelta(days=1)

        start = current_date.strftime("%Y-%m-%d")
        end = next_date.strftime("%Y-%m-%d")

        while True:
            print(
                f"{datetime.now().strftime("%Y-%m-%d %H:%M:%S")} Requesting {start} → {end}...",
                end=" "
            )

            data, status = get_reddit_comments(
                symbol,
                start,
                end,
                num_comments
            )

            if status == 200:
                comments = data["data"]
                months_comments.extend(comments)

                print(
                    f"OK — {len(comments)} comments | "
                    f"Total: {len(months_comments)}"
                )
                break

            else:
                print(
                    f"failed (status {status}). "
                    f"Retrying in 10 seconds..."
                )
                time.sleep(10)

        current_date = next_date

    print(
        f"\nFinished. Total comments: "
        f"{len(months_comments)}"
    )

    return months_comments

In [6]:
# Adaptive collecting function
# NOTE: (NOT in-use) this method seems to be super slow and ineffective

# * 429: rate limited → wait progressively longer, then retry the same window.
# * 422: server timeout → split the current time window in half.
# * Other errors: retry up to max_retries, then skip the window.
# * 200: accept the returned comments.

#                         Request
#                            │
#                  ┌─────────┼─────────┐
#                  │         │         │
#                200        422       429
#                  │         │         │
#               Accept      Split     Wait
#                data      window    and retry
#                            │
#                       ┌────┴────┐
#                       │         │
#                    Smaller   Smaller
#                    window    window
#
#                  Other status
#                       │
#                 ┌─────┴─────┐
#                 │           │
#             retries left   max retries
#                 │           │
#               Retry        Skip

def collect_reddit_comments_adaptive(
    symbol,
    start_date,
    end_date,
    num_comments=100,
    max_retries=6,
    min_window=timedelta(hours=1)
):
    months_comments = []

    # Create initial 1-day windows
    windows = []
    current_time = start_date

    while current_time < end_date:
        next_time = min(
            current_time + timedelta(days=1),
            end_date
        )
        windows.append((current_time, next_time))
        current_time = next_time

    while windows:

        current_time, next_time = windows.pop(0)

        attempts = 0

        while attempts < max_retries:
            attempts += 1

            start = current_time.strftime("%Y-%m-%d %H:%M")
            end = next_time.strftime("%Y-%m-%d %H:%M")

            timestamp = datetime.now().strftime("%H:%M:%S")

            data, status = get_reddit_comments(
                symbol,
                start,
                end,
                num_comments
            )

            # Success
            if status == 200:

                comments = data["data"]
                months_comments.extend(comments)

                print(
                    f"[{timestamp}] "
                    f"{start} → {end} | "
                    f"OK {len(comments)} | "
                    f"Total {len(months_comments)}"
                )

                break

            # Timeout
            elif status == 422:

                duration = next_time - current_time

                if duration <= min_window:

                    print(
                        f"[{timestamp}] "
                        f"{start} → {end} | "
                        f"422 → SKIP"
                    )

                    break

                midpoint = current_time + duration / 2

                print(
                    f"[{timestamp}] "
                    f"{start} → {end} | "
                    f"422 → SPLIT"
                )

                windows.insert(
                    0,
                    (midpoint, next_time)
                )
                windows.insert(
                    0,
                    (current_time, midpoint)
                )

                break

            # Rate limit
            elif status == 429:

                wait_time = 30 * attempts

                print(
                    f"[{timestamp}] "
                    f"{start} → {end} | "
                    f"429 → WAIT {wait_time}s"
                )

                time.sleep(wait_time)

            # Other errors
            else:

                if attempts < max_retries:

                    print(
                        f"[{timestamp}] "
                        f"{start} → {end} | "
                        f"{status} → RETRY {attempts}/{max_retries}"
                    )

                    time.sleep(10)

                else:

                    print(
                        f"[{timestamp}] "
                        f"{start} → {end} | "
                        f"{status} → SKIP"
                    )

    print(
        f"\nFinished | "
        f"Total comments: {len(months_comments)}"
    )

    return months_comments

In [7]:
#Collects Reddit comments using created_utc as a pagination cursor,
#retrying failed requests and moving backward in time until the specified start time is reached.
# NOTE: (use this) This is still slow, but the comment counts look healthy

# * 200: accept the returned comments, find the earliest `created_utc`,
#       and move the `before` timestamp backward to continue pagination.
# * 422: server timeout → wait briefly and retry the same time window.
# * 429: rate limited → wait until the rate-limit reset time, then retry.
# * Other errors: retry up to `max_retries`, then stop.
# * Continue until the `before` timestamp reaches the `after` timestamp.

#                         Request
#                            │
#                  ┌─────────┼─────────┐
#                  │         │         │
#                200        422       429
#                  │         │         │
#               Accept     Wait 5s    Wait headers.get("X-RateLimit-Reset")
#                data         │       and retry
#                  │          │
#                  │          └───────┐
#                  │                  │
#                  ▼                  │
#          Find minimum               │
#          created_utc                │
#                  │                  │
#                  ▼                  │
#          Update "before"            │
#          to minimum timestamp       │
#                  │                  │
#                  ▼                  │
#          Search earlier ────────────┘
#                  │
#                  ▼
#          until "before"
#          reaches "after"
#
#                 200 response
#                     │
#              ┌──────┴──────┐
#              │             │
#          Has comments   No comments
#              │             │
#              ▼             ▼
#        Process data      Stop
#              │
#              ▼
#       Find minimum
#       created_utc
#              │
#              ▼
#       Move "before"
#       backward
#
#                  Other status
#                       │
#                 ┌─────┴─────┐
#                 │           │
#             retries left   max retries
#                 │           │
#               Retry        Stop

def collect_reddit_comments_by_timestamp(
    symbol,
    after,
    before,
    num_comments=100,
    max_retries=6
):
    all_comments = []

    after_ts = int(
        pd.Timestamp(after, tz="UTC").timestamp()
    )

    current_before_ts = int(
        pd.Timestamp(before, tz="UTC").timestamp()
    )

    while current_before_ts > after_ts:

        start_str = pd.to_datetime(
            after_ts,
            unit="s",
            utc=True
        ).strftime("%Y-%m-%d %H:%M")

        before_str = pd.to_datetime(
            current_before_ts,
            unit="s",
            utc=True
        ).strftime("%Y-%m-%d %H:%M")

        attempts = 0

        while attempts < max_retries:

            attempts += 1

            data, status, headers = get_reddit_comments(
                symbol,
                start_str,
                before_str,
                num_comments
            )

            timestamp = datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            )

            # Success
            if status == 200:

                comments = data.get("data", []) if data else []

                if not comments:
                    print(
                        f"[{timestamp}] "
                        f"{start_str} → {before_str} | "
                        f"0 comments"
                    )
                    return all_comments

                all_comments.extend(comments)

                timestamps = [
                    c["created_utc"]
                    for c in comments
                    if c.get("created_utc") is not None
                ]

                if not timestamps:
                    return all_comments

                minimum_created_utc = min(timestamps)

                # Make sure cursor moves backward
                if minimum_created_utc >= current_before_ts:
                    print(
                        f"[{timestamp}] "
                        f"Cursor did not move backward."
                    )
                    return all_comments

                current_before_ts = minimum_created_utc

                earliest = pd.to_datetime(
                    minimum_created_utc,
                    unit="s",
                    utc=True
                ).strftime("%Y-%m-%d %H:%M")

                print(
                    f"[{timestamp}] "
                    f"{start_str} → {before_str} | "
                    f"{len(comments)} comments | "
                    f"earliest: {earliest} | "
                    f"total: {len(all_comments)}"
                )

                break

            # Server timeout
            elif status == 422:

                print(
                    f"[{timestamp}] "
                    f"{start_str} → {before_str} | "
                    f"422 timeout → wait 5s"
                )

                time.sleep(5)

            # Rate limit
            elif status == 429:

                reset = headers.get("X-RateLimit-Reset")

                try:
                    wait_time = min(max(float(reset), 1), 300)
                except (ValueError, TypeError):
                    wait_time = 30

                print(
                    f"[{timestamp}] HTTP 429 | "
                    f"X-RateLimit-Reset: {reset} | "
                    f"waiting {wait_time:.1f}s "
                    f"(attempt {attempts}/{max_retries})"
                )

                time.sleep(wait_time)

            # Other errors
            else:

                print(
                    f"[{timestamp}] "
                    f"{start_str} → {before_str} | "
                    f"HTTP {status} → wait 5s"
                )

                time.sleep(5)

        else:
            timestamp = datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            )

            # Skip the entire day containing the failed query
            failed_day = pd.to_datetime(
                current_before_ts,
                unit="s",
                utc=True
            ).normalize()

            next_before_ts = int(failed_day.timestamp())

            print(
                f"[{timestamp}] "
                f"{start_str} → {before_str} | "
                f"max retries reached; "
                f"skipping day {failed_day.strftime('%Y-%m-%d')}"
            )

            # Move the cursor to the beginning of the failed day
            if next_before_ts >= current_before_ts:
                next_before_ts = current_before_ts - 86400

            current_before_ts = next_before_ts
            continue

    timestamp = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    print(
        f"[{timestamp}] Finished | "
        f"{len(all_comments)} comments"
    )

    return all_comments

#### GME 1/4/2021 - 4/12/2021

In [23]:
url = "https://arctic-shift.photon-reddit.com/api/comments/search"

params = {
    "subreddit": "wallstreetbets",
    "body": "GME",
    "after": "2021-01-01",
    "before": "2021-02-01",
    "limit": 5
}

r = requests.get(url, params=params)

#data = r.json()
print("Request status (accepted: 200; rejected: 400):", r.status_code)
print("Type:",r.headers.get("Content-Type"))
print("Suggested reset wait time:",  r.headers.get("X-RateLimit-Reset"))
print(r.text[:500])

Request status (accepted: 200; rejected: 400): 522
Type: text/html; charset=UTF-8
Suggested reset wait time: None
<!DOCTYPE html>
<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>

<title>photon-reddit.com | 522: Connection timed out</title>
<meta charset="UTF-8" />
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta http-equiv="X-U


In [8]:
start_date = datetime(2021, 1, 4)
end_date = datetime(2021, 4, 12)

#months_comments = collect_reddit_comments("GME", start_date, end_date)
months_comments = collect_reddit_comments_by_timestamp("GME", start_date, end_date)

[2026-10-09 18:34:24] 2021-01-04 00:00 → 2021-04-12 00:00 | HTTP 522 → wait 5s
[2026-10-09 18:34:48] 2021-01-04 00:00 → 2021-04-12 00:00 | HTTP 522 → wait 5s
[2026-10-09 18:35:13] 2021-01-04 00:00 → 2021-04-12 00:00 | HTTP 522 → wait 5s
[2026-10-09 18:35:38] 2021-01-04 00:00 → 2021-04-12 00:00 | HTTP 522 → wait 5s
[2026-10-09 18:36:02] 2021-01-04 00:00 → 2021-04-12 00:00 | HTTP 522 → wait 5s


KeyboardInterrupt: 

In [ ]:
comments_clean = clean_comments(months_comments)
df = pd.DataFrame(comments_clean)
df.to_csv(
    f"{project_path}/wallstreetbets_GME_Jan4_2021_Apr12_2021.csv",
    index=False
)

#### AMC 1/4/2021 - 4/5/2021; 5/3/2021 - 7/5/2021


In [ ]:
start_date = datetime(2021, 1, 4)
end_date = datetime(2021, 4, 5)

months_comments = collect_reddit_comments_by_timestamp("AMC", start_date, end_date)

comments_clean = clean_comments(months_comments)
df = pd.DataFrame(comments_clean)
df.to_csv(
    f"{project_path}/wallstreetbets_AMC_Jan4_2021_Apr5_2021.csv",
    index=False
)

2021-01-04 00:00 → 2021-04-05 00:00 | 100 comments | earliest: 2021-04-04 18:16 | total: 100
2021-01-04 00:00 → 2021-04-04 18:16 | 422 timeout
2021-01-04 00:00 → 2021-04-04 18:16 | 422 timeout
2021-01-04 00:00 → 2021-04-04 18:16 | 100 comments | earliest: 2021-04-04 05:15 | total: 200
2021-01-04 00:00 → 2021-04-04 05:15 | 100 comments | earliest: 2021-04-03 18:16 | total: 300
2021-01-04 00:00 → 2021-04-03 18:16 | 100 comments | earliest: 2021-04-03 03:49 | total: 400
2021-01-04 00:00 → 2021-04-03 03:49 | 100 comments | earliest: 2021-04-02 19:47 | total: 500
2021-01-04 00:00 → 2021-04-02 19:47 | 100 comments | earliest: 2021-04-02 14:33 | total: 600
2021-01-04 00:00 → 2021-04-02 14:33 | 422 timeout
2021-01-04 00:00 → 2021-04-02 14:33 | 100 comments | earliest: 2021-04-02 08:06 | total: 700
2021-01-04 00:00 → 2021-04-02 08:06 | 100 comments | earliest: 2021-04-02 01:33 | total: 800
2021-01-04 00:00 → 2021-04-02 01:33 | 100 comments | earliest: 2021-04-01 22:26 | total: 900
2021-01-04 00

In [ ]:
# the above function is aborted, so we update the file name:

# 2021-01-04 00:00 → 2021-02-24 15:37 | 100 comments | earliest: 2021-02-24 15:30 | total: 63900
# 2021-01-04 00:00 → 2021-02-24 15:30 | 100 comments | earliest: 2021-02-24 15:21 | total: 64000
# 2021-01-04 00:00 → 2021-02-24 15:21 | 422 timeout
# 2021-01-04 00:00 → 2021-02-24 15:21 | 422 timeout
# 2021-01-04 00:00 → 2021-02-24 15:21 | 422 timeout
# 2021-01-04 00:00 → 2021-02-24 15:21 | 422 timeout
# 2021-01-04 00:00 → 2021-02-24 15:21 | 422 timeout
# 2021-01-04 00:00 → 2021-02-24 15:21 | 422 timeout
# Maximum retries reached.
# Finished | 64000 comments


df.to_csv(
    f"{project_path}/wallstreetbets_AMC_Feb24_2021_Apr5_2021.csv",
    index=False
)

In [ ]:
start_date = datetime(2021, 1, 4)
end_date = datetime(2021, 2, 25)

months_comments = collect_reddit_comments_by_timestamp("AMC", start_date, end_date)

comments_clean = clean_comments(months_comments)
df = pd.DataFrame(comments_clean)
df.to_csv(
    f"{project_path}/wallstreetbets_AMC_Jan4_2021_Feb25_2021.csv",
    index=False
)

[2026-10-08 23:54:42] 2021-01-04 00:00 → 2021-02-25 00:00 | 100 comments | earliest: 2021-02-24 23:52 | total: 100
[2026-10-08 23:54:45] 2021-01-04 00:00 → 2021-02-24 23:52 | 100 comments | earliest: 2021-02-24 23:43 | total: 200
[2026-10-08 23:54:49] 2021-01-04 00:00 → 2021-02-24 23:43 | 100 comments | earliest: 2021-02-24 23:35 | total: 300
[2026-10-08 23:54:51] 2021-01-04 00:00 → 2021-02-24 23:35 | 100 comments | earliest: 2021-02-24 23:26 | total: 400
[2026-10-08 23:54:54] 2021-01-04 00:00 → 2021-02-24 23:26 | 100 comments | earliest: 2021-02-24 23:16 | total: 500
[2026-10-08 23:54:57] 2021-01-04 00:00 → 2021-02-24 23:16 | 100 comments | earliest: 2021-02-24 23:07 | total: 600
[2026-10-08 23:54:59] 2021-01-04 00:00 → 2021-02-24 23:07 | 422 timeout → wait 5s
[2026-10-08 23:55:12] 2021-01-04 00:00 → 2021-02-24 23:07 | 100 comments | earliest: 2021-02-24 22:59 | total: 700
[2026-10-08 23:55:19] 2021-01-04 00:00 → 2021-02-24 22:59 | 100 comments | earliest: 2021-02-24 22:51 | total: 80

In [ ]:
start_date = datetime(2021, 5, 3)
end_date = datetime(2021, 7, 5)

months_comments = collect_reddit_comments_by_timestamp("AMC", start_date, end_date)

comments_clean = clean_comments(months_comments)
df = pd.DataFrame(comments_clean)
df.to_csv(
    f"{project_path}/wallstreetbets_AMC_May3_2021_Jul5_2021.csv",
    index=False
)

[2026-10-09 07:19:50] 2021-05-03 00:00 → 2021-07-05 00:00 | 422 timeout → wait 5s
[2026-10-09 07:19:58] 2021-05-03 00:00 → 2021-07-05 00:00 | 100 comments | earliest: 2021-07-04 20:16 | total: 100
[2026-10-09 07:19:59] 2021-05-03 00:00 → 2021-07-04 20:16 | 422 timeout → wait 5s
[2026-10-09 07:20:05] 2021-05-03 00:00 → 2021-07-04 20:16 | 422 timeout → wait 5s
[2026-10-09 07:20:10] 2021-05-03 00:00 → 2021-07-04 20:16 | 422 timeout → wait 5s
[2026-10-09 07:20:16] 2021-05-03 00:00 → 2021-07-04 20:16 | 422 timeout → wait 5s
[2026-10-09 07:20:22] 2021-05-03 00:00 → 2021-07-04 20:16 | 422 timeout → wait 5s
[2026-10-09 07:20:33] 2021-05-03 00:00 → 2021-07-04 20:16 | 100 comments | earliest: 2021-07-04 12:52 | total: 200
[2026-10-09 07:20:33] 2021-05-03 00:00 → 2021-07-04 12:52 | 422 timeout → wait 5s
[2026-10-09 07:20:45] 2021-05-03 00:00 → 2021-07-04 12:52 | 100 comments | earliest: 2021-07-03 20:15 | total: 300
[2026-10-09 07:20:50] 2021-05-03 00:00 → 2021-07-03 20:15 | 100 comments | earlie

#### BB 11/30/2020 - 12/9/2020; 1/13/2021 - 2/9/2021; 2/25/2021 - 6/21/2021


In [ ]:
start_date = datetime(2020, 11, 30)
end_date = datetime(2020, 12, 9)

months_comments = collect_reddit_comments_by_timestamp("BB", start_date, end_date)

comments_clean = clean_comments(months_comments)
df = pd.DataFrame(comments_clean)
df.to_csv(
    f"{project_path}/wallstreetbets_BB_Nov30_2020_Dec9_2020.csv",
    index=False
)

[2026-10-09 09:44:40] 2020-11-30 00:00 → 2020-12-09 00:00 | 100 comments | earliest: 2020-12-08 17:12 | total: 100
[2026-10-09 09:44:48] 2020-11-30 00:00 → 2020-12-08 17:12 | 100 comments | earliest: 2020-12-08 10:05 | total: 200
[2026-10-09 09:44:57] 2020-11-30 00:00 → 2020-12-08 10:05 | 100 comments | earliest: 2020-12-07 23:29 | total: 300
[2026-10-09 09:45:07] 2020-11-30 00:00 → 2020-12-07 23:29 | 100 comments | earliest: 2020-12-07 17:40 | total: 400
[2026-10-09 09:45:13] 2020-11-30 00:00 → 2020-12-07 17:40 | 100 comments | earliest: 2020-12-07 14:46 | total: 500
[2026-10-09 09:45:22] 2020-11-30 00:00 → 2020-12-07 14:46 | 100 comments | earliest: 2020-12-07 03:54 | total: 600
[2026-10-09 09:45:43] 2020-11-30 00:00 → 2020-12-07 03:54 | 100 comments | earliest: 2020-12-06 18:15 | total: 700
[2026-10-09 09:45:50] 2020-11-30 00:00 → 2020-12-06 18:15 | 100 comments | earliest: 2020-12-06 01:37 | total: 800
[2026-10-09 09:46:01] 2020-11-30 00:00 → 2020-12-06 01:37 | 422 timeout → wait 5

In [ ]:
start_date = datetime(2021, 1, 13)
end_date = datetime(2021, 2, 9)

months_comments = collect_reddit_comments_by_timestamp("BB", start_date, end_date)

comments_clean = clean_comments(months_comments)
df = pd.DataFrame(comments_clean)
df.to_csv(
    f"{project_path}/wallstreetbets_BB_Jan13_2021_Feb9_2021.csv",
    index=False
)

[2026-10-09 09:50:57] 2021-01-13 00:00 → 2021-02-09 00:00 | 100 comments | earliest: 2021-02-08 21:23 | total: 100
[2026-10-09 09:51:03] 2021-01-13 00:00 → 2021-02-08 21:23 | 100 comments | earliest: 2021-02-08 20:00 | total: 200
[2026-10-09 09:51:11] 2021-01-13 00:00 → 2021-02-08 20:00 | 100 comments | earliest: 2021-02-08 17:52 | total: 300
[2026-10-09 09:51:12] 2021-01-13 00:00 → 2021-02-08 17:52 | 422 timeout → wait 5s
[2026-10-09 09:51:22] 2021-01-13 00:00 → 2021-02-08 17:52 | 100 comments | earliest: 2021-02-08 16:06 | total: 400
[2026-10-09 09:51:24] 2021-01-13 00:00 → 2021-02-08 16:06 | 422 timeout → wait 5s
[2026-10-09 09:51:35] 2021-01-13 00:00 → 2021-02-08 16:06 | 100 comments | earliest: 2021-02-08 15:13 | total: 500
[2026-10-09 09:51:39] 2021-01-13 00:00 → 2021-02-08 15:13 | 100 comments | earliest: 2021-02-08 14:11 | total: 600
[2026-10-09 09:51:46] 2021-01-13 00:00 → 2021-02-08 14:11 | 100 comments | earliest: 2021-02-08 11:20 | total: 700
[2026-10-09 09:51:50] 2021-01-1

In [ ]:
start_date = datetime(2021, 2, 25)
end_date = datetime(2021, 6, 21)

months_comments = collect_reddit_comments_by_timestamp("BB", start_date, end_date)

comments_clean = clean_comments(months_comments)
df = pd.DataFrame(comments_clean)
df.to_csv(
    f"{project_path}/wallstreetbets_BB_Feb25_2021_Jun21_2021.csv",
    index=False
)

[2026-10-09 12:16:12] 2021-02-25 00:00 → 2021-06-21 00:00 | 422 timeout → wait 5s
[2026-10-09 12:16:18] 2021-02-25 00:00 → 2021-06-21 00:00 | 422 timeout → wait 5s
[2026-10-09 12:16:24] 2021-02-25 00:00 → 2021-06-21 00:00 | 422 timeout → wait 5s
[2026-10-09 12:16:36] 2021-02-25 00:00 → 2021-06-21 00:00 | 100 comments | earliest: 2021-06-20 10:57 | total: 100
[2026-10-09 12:16:37] 2021-02-25 00:00 → 2021-06-20 10:57 | 422 timeout → wait 5s
[2026-10-09 12:16:42] 2021-02-25 00:00 → 2021-06-20 10:57 | 422 timeout → wait 5s
[2026-10-09 12:16:48] 2021-02-25 00:00 → 2021-06-20 10:57 | 422 timeout → wait 5s
[2026-10-09 12:17:01] 2021-02-25 00:00 → 2021-06-20 10:57 | 100 comments | earliest: 2021-06-19 22:11 | total: 200
[2026-10-09 12:17:01] 2021-02-25 00:00 → 2021-06-19 22:11 | 422 timeout → wait 5s
[2026-10-09 12:17:12] 2021-02-25 00:00 → 2021-06-19 22:11 | 100 comments | earliest: 2021-06-19 12:42 | total: 300
[2026-10-09 12:17:13] 2021-02-25 00:00 → 2021-06-19 12:42 | 422 timeout → wait 5s

#### SMR 2/19/2024 - 4/15/2024; 9/2/2024 - 4/14/2025 (Have nothing so far)

In [ ]:
start_date = datetime(2024, 2, 19)
end_date = datetime(2024, 4, 15)

months_comments = collect_reddit_comments_by_timestamp("SMR", start_date, end_date)

comments_clean = clean_comments(months_comments)
df = pd.DataFrame(comments_clean)
df.to_csv(
    f"{project_path}/wallstreetbets_SMR_Feb19_2024_Apr15_2024.csv",
    index=False
)

[2026-10-09 15:16:33] 2024-02-19 00:00 → 2024-04-15 00:00 | 422 timeout → wait 5s
[2026-10-09 15:16:48] 2024-02-19 00:00 → 2024-04-15 00:00 | 422 timeout → wait 5s
[2026-10-09 15:17:04] 2024-02-19 00:00 → 2024-04-15 00:00 | 422 timeout → wait 5s
[2026-10-09 15:17:10] 2024-02-19 00:00 → 2024-04-15 00:00 | 422 timeout → wait 5s
[2026-10-09 15:17:16] 2024-02-19 00:00 → 2024-04-15 00:00 | 422 timeout → wait 5s
[2026-10-09 15:17:32] 2024-02-19 00:00 → 2024-04-15 00:00 | 422 timeout → wait 5s
[2026-10-09 15:17:37] 2024-02-19 00:00 → 2024-04-15 00:00 | max retries reached; skipping day 2024-04-15
[2026-10-09 15:17:48] 2024-02-19 00:00 → 2024-04-14 00:00 | 422 timeout → wait 5s
[2026-10-09 15:17:54] 2024-02-19 00:00 → 2024-04-14 00:00 | 422 timeout → wait 5s
[2026-10-09 15:18:01] 2024-02-19 00:00 → 2024-04-14 00:00 | 422 timeout → wait 5s
[2026-10-09 15:18:06] 2024-02-19 00:00 → 2024-04-14 00:00 | 422 timeout → wait 5s
[2026-10-09 15:18:13] 2024-02-19 00:00 → 2024-04-14 00:00 | 422 timeout → w

KeyboardInterrupt: 

In [ ]:
start_date = datetime(2024, 9, 2)
end_date = datetime(2025, 4, 14)

months_comments = collect_reddit_comments_by_timestamp("SMR", start_date, end_date)

comments_clean = clean_comments(months_comments)
df = pd.DataFrame(comments_clean)
df.to_csv(
    f"{project_path}/wallstreetbets_SMR_Feb19_2024_Apr15_2024.csv",
    index=False
)

[2026-10-09 16:08:17] 2024-09-02 00:00 → 2025-04-14 00:00 | HTTP 522 → wait 5s


KeyboardInterrupt: 

## Other Tools?